# Russian Edit Corrector Pipeline

Top-to-bottom entrypoint for config loading, environment checks, balanced dataset rebuilding, label smoke checks, evaluation reports, and manual examples.


In [ ]:
from pathlib import Path
import os
import sys

if os.environ.get('RUSSIAN_CORRECTOR_DISABLE_MODEL_TRAINING') != '1':
    os.environ.setdefault('RUSSIAN_CORRECTOR_RUN_MODEL_TRAINING', '1')

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src').exists() else cwd.parent
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import load_config

config_path = project_root / 'configs' / 'config.yaml'
config = load_config(config_path)
config['project']['name']


In [ ]:
import sys
import torch

env = {
    'python': sys.version.split()[0],
    'cuda_available': torch.cuda.is_available(),
    'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
}
env


In [ ]:
import importlib
import src.data.full_dataset_builder as fdb

importlib.reload(fdb)

dataset_build = fdb.build_dataset_from_config(config, force=True)
dataset_build


In [ ]:
import json
from collections import Counter
import pandas as pd

processed_path = Path(config['data']['processed_train_path'])
dataset_frame = pd.read_csv(processed_path)
expected_total = int(os.environ.get('RUSSIAN_CORRECTOR_DATASET_LIMIT') or config['data']['target_total_examples'])
assert len(dataset_frame) == expected_total, (len(dataset_frame), expected_total)

def parse_error_types(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    parsed = json.loads(value) if str(value).strip() else []
    return parsed if isinstance(parsed, list) else [parsed]

error_type_counts = Counter(
    error_type
    for value in dataset_frame['error_types']
    for error_type in parse_error_types(value)
)
summary = {
    'shape': dataset_frame.shape,
    'splits': dataset_frame['split'].value_counts().to_dict(),
    'composition': {
        'clean': int(dataset_frame['is_clean'].astype(bool).sum()),
        'synthetic': int((dataset_frame['is_synthetic'].astype(bool) & ~dataset_frame['is_clean'].astype(bool)).sum()),
        'real': int((~dataset_frame['is_synthetic'].astype(bool) & ~dataset_frame['is_clean'].astype(bool)).sum()),
    },
    'error_type_counts': dict(sorted(error_type_counts.items())),
}
summary


In [ ]:
from src.alignment.aligner import Aligner

examples = dataset_frame.head(6).to_dict('records')
aligner = Aligner()
[(row['source'], aligner.align(row['source'], row['target']).is_supported) for row in examples]


In [ ]:
from src.training.train import train

train_result = train(config_path)
train_result


In [ ]:
from src.inference.corrector import Corrector
from src.inference.model_corrector import TrainedModelCorrector

adapter_dir = project_root / config['paths']['adapter_output_dir']
heads_path = project_root / config['paths']['heads_output_dir'] / 'heads.pt'
use_trained_corrector = bool(train_result.get('model_training_ran')) and adapter_dir.exists() and heads_path.exists()
model_corrector = TrainedModelCorrector.from_config(config) if use_trained_corrector else Corrector()
corrector_kind = 'trained_model' if use_trained_corrector else 'rule_fallback'
train_result.get('evaluation_split'), corrector_kind, train_result.get('evaluation_metrics')


In [ ]:
manual = ['Я незнаю что делать', 'сегодня что то произошло', 'Я люблю этот дом']
corrector_kind, [(text, model_corrector.correct(text).corrected_text) for text in manual]
